### Build a human-only Ensembl ↔ NCBI Gene RefSeq-summary table.

Output columns:
- ensembl_id  
- ncbi_gene_id  
- symbol  
- synonyms  
- refseq_summary  

Sources downloaded from official NCBI Gene FTP:
- gene_summary.gz  
- gene2ensembl.gz  
- GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz  

#### Usage:
    python build_human_ncbi_refseq_summary_tsv.py --out human_ensembl_ncbi_refseq_summary_synonyms.tsv

#### CLI:

python build_human_ncbi_refseq_summary_tsv.py --cache-dir ncbi_gene_cache --out human_ensembl_ncbi_refseq_summary_synonyms.tsv
"""

In [6]:
## #!/usr/bin/env python3

In [11]:
from __future__ import annotations

# import argparse
import os
import csv
import gzip
import sys
import urllib.request
from pathlib import Path
from typing import Dict, Iterable, List, Set

In [12]:
NCBI_URLS = {
    "gene_summary": "https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene_summary.gz",
    "gene2ensembl": "https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2ensembl.gz",
    "human_gene_info": (
        "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/"
        "Homo_sapiens.gene_info.gz"
    ),
}

HUMAN_TAX_ID = "9606"

In [19]:
def download_if_missing(url: str, path: Path) -> None:
    """Download a file only if it is not already present."""
    if path.exists() and path.stat().st_size > 0:
        print(f"[OK] Already exists: {path}")
        return

    print(f"[DOWNLOAD] {url}")
    path.parent.mkdir(parents=True, exist_ok=True)

    with urllib.request.urlopen(url) as response, open(path, "wb") as out:
        total = int(response.headers.get("Content-Length", 0) or 0)
        downloaded = 0

        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break

            out.write(chunk)
            downloaded += len(chunk)

            if total:
                pct = downloaded / total * 100
                print(f"\r  {downloaded / 1e6:.1f} MB / {total / 1e6:.1f} MB ({pct:.1f}%)", end="")
            else:
                print(f"\r  {downloaded / 1e6:.1f} MB", end="")

    print(f"\n[SAVED] {path}")


def open_gzip_tsv(path: Path):
    """Open a gzipped TSV file as text."""
    return gzip.open(path, "rt", encoding="utf-8", errors="replace", newline="")


def normalize_empty(value: str | None) -> str:
    if value is None:
        return ""
    value = str(value).strip()
    if value in {"", "-", "NA", "N/A"}:
        return ""
    return value


def normalize_synonyms(value: str | None) -> str:
    """NCBI gene_info synonyms are pipe-separated. Convert to '; '."""
    value = normalize_empty(value)
    if not value:
        return ""

    parts = []
    seen = set()

    for item in value.split("|"):
        item = item.strip()
        if not item or item in seen:
            continue
        seen.add(item)
        parts.append(item)

    return "; ".join(parts)


def load_human_gene_info(path: Path) -> Dict[str, dict]:
    """
    Load human gene symbols and synonyms from Homo_sapiens.gene_info.gz.

    Returns:
        gene_id -> {
            "symbol": ...,
            "synonyms": ...,
            "description": ...
        }
    """
    print(f"[READ] {path}")

    result: Dict[str, dict] = {}

    with open_gzip_tsv(path) as handle:
        reader = csv.DictReader(handle, delimiter="\t")

        # Header usually starts with '#tax_id'
        fieldnames = reader.fieldnames or []
        if "#tax_id" in fieldnames:
            tax_col = "#tax_id"
        elif "tax_id" in fieldnames:
            tax_col = "tax_id"
        else:
            raise ValueError(f"Could not find tax_id column in {path}. Found: {fieldnames}")

        for row in reader:
            if row.get(tax_col) != HUMAN_TAX_ID:
                continue

            gene_id = normalize_empty(row.get("GeneID"))
            if not gene_id:
                continue

            result[gene_id] = {
                "symbol": normalize_empty(row.get("Symbol")),
                "synonyms": normalize_synonyms(row.get("Synonyms")),
                "description": normalize_empty(row.get("description")),
            }

    print(f"[OK] Human gene_info records: {len(result):,}")
    return result


def load_human_gene_summaries(path: Path) -> Dict[str, str]:
    """
    Load RefSeq/NCBI Gene summaries from gene_summary.gz.

    Returns:
        gene_id -> summary
    """
    print(f"[READ] {path}")

    summaries: Dict[str, str] = {}

    with open_gzip_tsv(path) as handle:
        reader = csv.DictReader(handle, delimiter="\t")

        fieldnames = reader.fieldnames or []
        if "#tax_id" in fieldnames:
            tax_col = "#tax_id"
        elif "tax_id" in fieldnames:
            tax_col = "tax_id"
        else:
            raise ValueError(f"Could not find tax_id column in {path}. Found: {fieldnames}")

        # Usually: #tax_id, GeneID, Summary
        summary_col = "Summary" if "Summary" in fieldnames else "summary"

        for row in reader:
            if row.get(tax_col) != HUMAN_TAX_ID:
                continue

            gene_id = normalize_empty(row.get("GeneID"))
            summary = normalize_empty(row.get(summary_col))

            if gene_id and summary:
                summaries[gene_id] = summary

    print(f"[OK] Human gene summaries: {len(summaries):,}")
    return summaries


def load_human_gene2ensembl(path: Path) -> Dict[str, Set[str]]:
    """
    Load NCBI GeneID -> Ensembl gene ID mappings from gene2ensembl.gz.

    Returns:
        gene_id -> set(ensembl_gene_ids)
    """
    print(f"[READ] {path}")

    mapping: Dict[str, Set[str]] = {}

    with open_gzip_tsv(path) as handle:
        reader = csv.DictReader(handle, delimiter="\t")

        fieldnames = reader.fieldnames or []
        if "#tax_id" in fieldnames:
            tax_col = "#tax_id"
        elif "tax_id" in fieldnames:
            tax_col = "tax_id"
        else:
            raise ValueError(f"Could not find tax_id column in {path}. Found: {fieldnames}")

        ensembl_col_candidates = [
            "Ensembl_gene_identifier",
            "Ensembl_gene",
            "ensembl_gene_id",
        ]

        ensembl_col = None
        for candidate in ensembl_col_candidates:
            if candidate in fieldnames:
                ensembl_col = candidate
                break

        if ensembl_col is None:
            raise ValueError(
                f"Could not find Ensembl gene column in {path}. Found: {fieldnames}"
            )

        for row in reader:
            if row.get(tax_col) != HUMAN_TAX_ID:
                continue

            gene_id = normalize_empty(row.get("GeneID"))
            ensembl_id = normalize_empty(row.get(ensembl_col))

            if not gene_id or not ensembl_id:
                continue

            mapping.setdefault(gene_id, set()).add(ensembl_id)

    print(f"[OK] Human GeneID→Ensembl mappings: {len(mapping):,}")
    return mapping


def write_output(
    out_path: Path,
    gene_info: Dict[str, dict],
    summaries: Dict[str, str],
    gene2ensembl: Dict[str, Set[str]],
    keep_without_summary: bool = False,
) -> None:
    """Write final TSV."""
    print(f"[WRITE] {out_path}")

    out_path.parent.mkdir(parents=True, exist_ok=True)

    n_rows = 0
    n_with_summary = 0

    with open(out_path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            delimiter="\t",
            fieldnames=[
                "ensembl_id",
                "ncbi_gene_id",
                "symbol",
                "synonyms",
                "refseq_summary",
            ],
        )
        writer.writeheader()

        for gene_id in sorted(gene2ensembl, key=lambda x: int(x) if x.isdigit() else x):
            summary = summaries.get(gene_id, "")

            if not summary and not keep_without_summary:
                continue

            info = gene_info.get(gene_id, {})
            symbol = info.get("symbol", "")
            synonyms = info.get("synonyms", "")

            for ensembl_id in sorted(gene2ensembl[gene_id]):
                writer.writerow(
                    {
                        "ensembl_id": ensembl_id,
                        "ncbi_gene_id": gene_id,
                        "symbol": symbol,
                        "synonyms": synonyms,
                        "refseq_summary": summary,
                    }
                )
                n_rows += 1
                if summary:
                    n_with_summary += 1

    print(f"[OK] Rows written: {n_rows:,}")
    print(f"[OK] Rows with RefSeq/NCBI summary: {n_with_summary:,}")




In [40]:
ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_NCBI = ROOT0 / "data/ncbi"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)
print("ROOT_NCBI:", ROOT_NCBI)

import pandas as pd
from libs.Basic import create_dir, pdreadcsv, pdwritecsv

os.listdir(ROOT_NCBI)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src
ROOT_NCBI: /home/flavio/uv/perturb_agent/data/ncbi


['.~lock.human_ensembl_ncbi_refseq_summary_synonyms.tsv#',
 'hugo_gene_table_ensembl_symbol_name_description_uniprot.tsv',
 '.~lock.hugo_gene_table_ensembl_symbol_name_description_uniprot.tsv#',
 'human_ensembl_ncbi_refseq_summary_synonyms.tsv',
 'ncbi_gene_cache',
 'ensembl_id_ncbi_refseq_description.tsv']

In [21]:
fname = "human_ensembl_ncbi_refseq_summary_synonyms.tsv"
filename = ROOT_NCBI / fname
out_path=Path(filename)

root_cache = create_dir(ROOT_NCBI / "ncbi_gene_cache")


In [22]:
paths = {key: root_cache / Path(url).name for key, url in NCBI_URLS.items()}
paths

{'gene_summary': PosixPath('/home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/gene_summary.gz'),
 'gene2ensembl': PosixPath('/home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/gene2ensembl.gz'),
 'human_gene_info': PosixPath('/home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/Homo_sapiens.gene_info.gz')}

In [23]:
for key, url in NCBI_URLS.items():
    download_if_missing(url, paths[key])

[DOWNLOAD] https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene_summary.gz
  22.2 MB / 22.2 MB (100.0%)
[SAVED] /home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/gene_summary.gz
[DOWNLOAD] https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2ensembl.gz
  289.3 MB / 289.3 MB (100.0%)
[SAVED] /home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/gene2ensembl.gz
[DOWNLOAD] https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz
  5.2 MB / 5.2 MB (100.0%)
[SAVED] /home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/Homo_sapiens.gene_info.gz


In [24]:
gene_info = load_human_gene_info(paths["human_gene_info"])
summaries = load_human_gene_summaries(paths["gene_summary"])
gene2ensembl = load_human_gene2ensembl(paths["gene2ensembl"])

[READ] /home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/Homo_sapiens.gene_info.gz
[OK] Human gene_info records: 193,794
[READ] /home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/gene_summary.gz
[OK] Human gene summaries: 151,658
[READ] /home/flavio/uv/perturb_agent/data/ncbi/ncbi_gene_cache/gene2ensembl.gz
[OK] Human GeneID→Ensembl mappings: 38,516


In [26]:
gene_info['1']

{'symbol': 'A1BG',
 'synonyms': 'A1B; ABG; GAB; HYST2477',
 'description': 'alpha-1-B glycoprotein'}

In [27]:
write_output(
    out_path=out_path,
    gene_info=gene_info,
    summaries=summaries,
    gene2ensembl=gene2ensembl,
    keep_without_summary=False,
)


[WRITE] /home/flavio/uv/perturb_agent/data/ncbi/human_ensembl_ncbi_refseq_summary_synonyms.tsv
[OK] Rows written: 22,503
[OK] Rows with RefSeq/NCBI summary: 22,503


### Merging both tables

In [28]:
fname_hugo = "hugo_gene_table_ensembl_symbol_name_description_uniprot.tsv"
fname_refseq = "human_ensembl_ncbi_refseq_summary_synonyms.tsv"

In [35]:
ROOT_NCBI

PosixPath('/home/flavio/uv/perturb_agent/data/ncbi')

In [37]:
dfh = pdreadcsv(fname_hugo, ROOT_NCBI)
print(len(dfh))
dfh.columns

39439


Index(['ensembl_id', 'symbol', 'name', 'uniprot_id'], dtype='object')

In [38]:
dfref = pdreadcsv(fname_refseq, ROOT_NCBI)
print(len(dfref))
dfref.columns

22503


Index(['ensembl_id', 'ncbi_gene_id', 'symbol', 'synonyms', 'refseq_summary'], dtype='object')

In [39]:
len(dfh), len(dfref)

(39439, 22503)

In [46]:
dfn = merged_df = pd.merge(dfh, dfref, on=['ensembl_id', 'symbol'], how='outer')
dfn = dfn.drop_duplicates('ensembl_id')
print(len(dfn))
dfn.columns


39740


Index(['ensembl_id', 'symbol', 'name', 'uniprot_id', 'ncbi_gene_id',
       'synonyms', 'refseq_summary'],
      dtype='object')

In [47]:
fname = 'hugo_gene_table_refseq_uniprot.tsv'
pdwritecsv(dfn, fname, ROOT_NCBI)

True

In [48]:
dfn.head(3)

,ensembl_id,symbol,name,uniprot_id,ncbi_gene_id,synonyms,refseq_summary
0,ENSG00000000003,TSPAN6,tetraspanin 6,O43657,7105.0,T245; TM4SF6; TSPAN-6,The protein encoded by this gene is a member o...
1,ENSG00000000005,TNMD,tenomodulin,Q9H2S6,64102.0,BRICD4; CHM1L; TEM,This gene encodes a protein that is related to...
2,ENSG00000000419,DPM1,dolichyl-phosphate mannosyltransferase subunit...,O60762,8813.0,CDGIE; MPDS,Dolichol-phosphate mannose (Dol-P-Man) serves ...
